# Train the critic / value decoder — per-player win-rate + rank pretrain

Pretrains a **per-player `ValueDecoder`** on the V2 `player_state` produced by the cross_entity stack. The backbone (L2 cross-entity attention + `PlayerConsolidator`, plus the frozen L0/L1 encoders) is **warm-started from a V2 `playerpair` checkpoint**. This run trains the fresh value decoder at full LR and fine-tunes L2 at 0.1x LR; `PlayerConsolidator` stays frozen. This is the **PPO-critic precursor**: it learns to map a game state to each player's probability of winning.

**`--head-set critic`.** Passing `--head-set critic` (below) selects `CrossEntityCriticModel` (the V2 backbone + a `ValueDecoder` that emits a per-player `value_logit` of shape `(B, 4)`). `--init-from $V2_CKPT` loads the V2 playerpair backbone (`cross.*` / `consolidator.*`) with `strict=False`; `--unfreeze l2 --backbone-lr-mult 0.1` trains `value_decoder.*` at `LR` and `cross.*` at `0.1 * LR`, while keeping `consolidator.*` frozen.

**Losses.** Two terms on the per-player `value_logit`:
- **Win-rate BCE** — per-player binary cross-entropy with the winner labelled `1` and the other seats `0`. **Tie rows are excluded** (tie → all-zero target, masked out) and padding seats are masked via `player_valid`.
- **ListMLE ranking** (Plackett-Luce) on **`final_rank`** — the final standing derived from elimination order + final ship counts. Weighted by **λ_rank = 0.5** (`CRITIC_LAMBDA_RANK`).

At inference the deployed value is **`V(s) = sigmoid(value_logit[:, 0])` = P(learner wins)** — the learner occupies seat 0.

> **Cache requirement.** This pretrain needs a cross_entity cache **rebuilt with the new `final_rank` / `player_valid` columns** — the critic loss asserts `player_valid` is present and non-zero (all-zero → it raises and tells you to rebuild). An older cache will fail fast.

**Prior/future windows:** prior input uses `HISTORY_OFFSETS = (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)`, so the deployable view sees 10 slots from `t-45 ... t`. With `--aux-posterior`, the teacher view uses the mirrored future window `t+45, t+40, ..., t` (also 10 slots); only the prior view is used at deployment.

**Data path: prebuilt `.pt` cache.** Pulls `cross_entity_cache_{tiny,100k,full}.pt` (`DATASET` selector below; default `'100k'`, the ~21 GB balanced sample) plus the code/weights bundle and `manifest.json`. If a chunk manifest exists (`<cache>.manifest.json` or `<cache-stem>.manifest.json`), the cache chunks are downloaded in parallel and reassembled; otherwise the notebook falls back to the single `.pt` object. `train()` slices the cache into 80/10/10 train/val/test via the manifest's per-split stem lists — no CSV walk.

## Inputs from GCS

```
gs://orbit-wars-shipping/cross_entity/
  code.tgz                       # agents/ + scripts/
  weights.tgz                    # frozen L0 (planet, fleet, comet) d=256
  cross_entity_cache_tiny.pt     # smoke cache (few episodes)
  cross_entity_cache_100k.pt     # ~21 GB balanced sample (default)
  cross_entity_cache_full.pt     # all episodes
  entity_encoder_best.pt         # frozen L1 (May-21 baseline)
  manifest.json                  # split lists
  runs/cross_T10_v2_playerpair_d256_b512_10ep_lr0.0001_20260529-061744/
    cross_entity_best.pt         # V2 playerpair backbone (warm-start; L2 fine-tuned)
```


## 0. Config

In [ ]:
BATCH_SIZE  = 256
EPOCHS      = 10
LR          = 5e-4   # decoder; L2 fine-tunes at 0.1x = 5e-5 (smaller, anti-overfit)
NUM_WORKERS = 2
NUM_LOAD_WORKERS = 8

# The critic reads a prebuilt .pt cache. Pick the corpus size. The cache MUST
# have been rebuilt with the new final_rank / player_valid / delta columns
# (the critic asserts player_valid != 0). 'critic30k' is the fast-first-run
# sample (34,216 snapshots, 6.6 GB, 115/20/15 episode split) and the ONLY cache
# rebuilt with the new labels so far — the 100k/full/tiny caches on GCS are
# STALE (no labels) and will trip the player_valid assertion if selected.
#   critic30k -> cross_entity_cache_critic30k.pt  (fast first run, labelled)
#   tiny      -> cross_entity_cache_tiny.pt       (end-to-end smoke, STALE)
#   100k      -> cross_entity_cache_100k.pt       (~21 GB, STALE: no labels)
#   full      -> cross_entity_cache_full.pt       (all episodes, STALE)
DATASET = 'critic30k'
DATASET_CACHE = {
    'critic30k': 'cross_entity_cache_critic30k.pt',
    'tiny': 'cross_entity_cache_tiny.pt',
    '100k': 'cross_entity_cache_100k.pt',
    'full': 'cross_entity_cache_full.pt',
}
CACHE_OBJECT = DATASET_CACHE[DATASET]
print(f'batch={BATCH_SIZE}  epochs={EPOCHS}  lr={LR}  workers={NUM_WORKERS}  load_workers={NUM_LOAD_WORKERS}  dataset={DATASET} ({CACHE_OBJECT})')

## 1. Authenticate + pull bundle

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/cross_entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, shutil, subprocess, time, concurrent.futures, json
from pathlib import Path

WORK = Path('/content/orbit-wars')

# V2 playerpair backbone the critic warm-starts from. This notebook fine-tunes
# L2 at a small LR while keeping PlayerConsolidator frozen; staged + exposed
# as V2_CKPT below.
V2_CKPT_SRC = ('gs://orbit-wars-shipping/cross_entity/runs/'
               'cross_T10_v2_playerpair_d256_b512_10ep_lr0.0001_20260529-061744/'
               'cross_entity_best.pt')
V2_CKPT_LOCAL = WORK / 'cross_entity_v2_playerpair_best.pt'
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Wipe stale extracted state before parallel staging starts.
for rel in ('agents', 'scripts', 'ckpts', 'data/datasets', 'data/runs'):
    shutil.rmtree(WORK / rel, ignore_errors=True)
(WORK / 'data/datasets').mkdir(parents=True, exist_ok=True)
for rel in ('cross_entity', 'entity', 'fleet', 'planet'):
    (WORK / 'data/datasets' / rel).mkdir(parents=True, exist_ok=True)

# Cache lands next to the cross_entity CSVs so the manifest sits beside it.
CACHE_PATH = WORK / 'data/datasets/cross_entity' / CACHE_OBJECT

def cp(src, dst, *, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} → {dst.name} ...', flush=True)
    subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
    return dst.name, time.time() - t0, dst.stat().st_size

def pull_cache():
    """Pull CACHE_OBJECT. Prefer chunk manifest for parallel cache download."""
    manifest_local = WORK / f'{CACHE_OBJECT}.manifest.json'
    if manifest_local.exists():
        manifest_local.unlink()
    manifest_candidates = [
        f'{BUCKET}/{CACHE_OBJECT}.manifest.json',
        f'{BUCKET}/{Path(CACHE_OBJECT).stem}.manifest.json',
    ]
    manifest_obj = None
    for cand in manifest_candidates:
        try:
            subprocess.run(
                ['gcloud', 'storage', 'cp', cand, str(manifest_local)],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
            manifest_obj = cand
            break
        except subprocess.CalledProcessError:
            continue
    if manifest_obj is None:
        print(f'  no chunk manifest for {CACHE_OBJECT}; pulling single object', flush=True)
        return cp(f'{BUCKET}/{CACHE_OBJECT}', CACHE_PATH)

    manifest = json.loads(manifest_local.read_text())
    chunks = manifest.get('chunks') or manifest.get('parts') or []
    if not chunks:
        print(f'  empty chunk manifest for {CACHE_OBJECT}; pulling single object', flush=True)
        return cp(f'{BUCKET}/{CACHE_OBJECT}', CACHE_PATH)

    part_dir = WORK / 'cache_parts'
    shutil.rmtree(part_dir, ignore_errors=True)
    part_dir.mkdir(parents=True, exist_ok=True)
    max_part_workers = max(1, min(int(NUM_LOAD_WORKERS), len(chunks)))
    print(f'  pulling {len(chunks)} cache chunks with {max_part_workers} workers ...', flush=True)
    t0 = time.time()

    def pull_part(spec):
        name = spec['name']
        src = name if str(name).startswith('gs://') else f'{BUCKET}/{name}'
        dst = part_dir / Path(name).name
        subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
        expected = int(spec.get('bytes', dst.stat().st_size))
        actual = dst.stat().st_size
        if actual != expected:
            raise RuntimeError(f'chunk size mismatch for {name}: {actual} != {expected}')
        return dst, actual

    parts = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_part_workers) as pool:
        futs = [pool.submit(pull_part, spec) for spec in chunks]
        for fut in concurrent.futures.as_completed(futs):
            parts.append(fut.result())

    if CACHE_PATH.exists():
        CACHE_PATH.unlink()
    with CACHE_PATH.open('wb') as out:
        for spec in chunks:
            part = part_dir / Path(spec['name']).name
            with part.open('rb') as fh:
                shutil.copyfileobj(fh, out, length=64 * 1024 * 1024)
    actual = CACHE_PATH.stat().st_size
    expected_total = int(manifest.get('total_bytes', actual))
    if actual != expected_total:
        raise RuntimeError(f'cache reassembly size mismatch: {actual} != {expected_total}')
    return CACHE_PATH.name, time.time() - t0, actual

def pull_and_stage(src, dst, stage):
    if stage == 'cache':
        name, dt, size = pull_cache()
    else:
        name, dt, size = cp(src, dst)
    t0 = time.time()
    if stage in ('code', 'weights'):
        subprocess.run(['tar', 'xzf', str(dst)], check=True)
    return name, dt, size, stage, time.time() - t0

TASKS = [
    (f'{BUCKET}/code.tgz',                  WORK / 'code.tgz',                 'code'),
    (f'{BUCKET}/weights.tgz',               WORK / 'weights.tgz',              'weights'),
    (f'{BUCKET}/entity_encoder_best.pt',    WORK / 'entity_encoder_best.pt',   'file'),
    (V2_CKPT_SRC,                           V2_CKPT_LOCAL,                     'file'),
    (f'{BUCKET}/manifest.json',             WORK / 'manifest.json',            'file'),
    (f'{BUCKET}/{CACHE_OBJECT}',            CACHE_PATH,                        'cache'),
]
T_START = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(TASKS)) as pool:
    futs = [pool.submit(pull_and_stage, s, d, stage) for s, d, stage in TASKS]
    for f in concurrent.futures.as_completed(futs):
        results.append(f.result())
for name, dt, size, stage, extract_dt in sorted(results, key=lambda t: -t[2]):
    size_msg = ' streamed' if size < 0 else f'{size / 1024 / 1024:>9.1f} MB'
    print(f'  {name:<34s} {size_msg}  pull={dt:5.1f}s  stage={extract_dt:5.1f}s  {stage}')
# Manifest lives next to the cross_entity cache.
(WORK / 'data/datasets/cross_entity').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'manifest.json', WORK / 'data/datasets/cross_entity/manifest.json')
print(f'\ncache: {CACHE_PATH}')
print(f'total wall: {time.time()-T_START:.1f}s')

In [ ]:
# Parallel staging happened in the previous cell. This cell only clears
# stale imports and prints a quick filesystem sanity check.
import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!du -sh data/datasets/*
!ls -la

## 1b. Verify the critic model builds + cache has final_rank / player_valid

In [ ]:
import agents
from agents.transformer_v2.history import HISTORY_OFFSETS, N_HISTORY
print(f'agents module: {agents.__file__}')
print(f'HISTORY_OFFSETS: {HISTORY_OFFSETS}')
assert HISTORY_OFFSETS == (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)
assert N_HISTORY == 10

import torch
# The critic = V2 backbone + per-player ValueDecoder (value_logit shape (B, 4)).
from agents.transformer_v2.pretrain.cross_entity import CrossEntityCriticModel
m = CrossEntityCriticModel(d_model=256)
print(f'CrossEntityCriticModel OK: params={sum(p.numel() for p in m.parameters()):,}')

# Cache must have been rebuilt with the new final_rank / player_valid columns.
# Load one item and assert the critic-label keys are present with shape (4,).
from agents.transformer_v2.pretrain.cross_entity import CachedCrossEntitySnapshotDataset
ds = CachedCrossEntitySnapshotDataset(str(CACHE_PATH))
print(f'cache items: {len(ds):,}  (from {CACHE_PATH.name})')
sample = ds[0]
missing = [k for k in ('final_rank', 'player_valid') if k not in sample]
assert not missing, (
    f'cache is missing {missing} -> it predates the final_rank/player_valid '
    f'columns. Rebuild the cross_entity cache with the updated featurizer '
    f"before training head_set='critic'."
)
assert tuple(sample['final_rank'].shape) == (4,), sample['final_rank'].shape
assert tuple(sample['player_valid'].shape) == (4,), sample['player_valid'].shape
assert float(sample['player_valid'].sum()) > 0.0, (
    'player_valid is all-zero -> cache has critic keys but was likely rebuilt '
    'from stale CSVs. Rebuild cross_entity cache after regenerating the CSVs.'
)
print(f"final_rank={sample['final_rank'].tolist()}  "
      f"player_valid={sample['player_valid'].tolist()}  -> cache OK for critic")
del ds, sample
import gc
gc.collect()

## 2. Verify GPU

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 3. Stage L0 + L1 ckpts

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
ENTITY_RUN_DIR = Path('/content/orbit-wars/ckpts/entity')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, ENTITY_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/entity_encoder_best.pt', ENTITY_RUN_DIR / 'entity_encoder_best.pt')

# Stage the V2 playerpair backbone the critic warm-starts from. Pulled in the
# bundle cell to V2_CKPT_LOCAL; copy it next to the other ckpts and expose the
# V2_CKPT path passed to --init-from below.
V2_CKPT = Path('/content/orbit-wars/ckpts/cross_entity_v2_playerpair_best.pt')
shutil.copy(str(V2_CKPT_LOCAL), V2_CKPT)

import torch
for tag, p in (('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
                ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
                ('entity', ENTITY_RUN_DIR / 'entity_encoder_best.pt')):
    c = torch.load(p, map_location='cpu', weights_only=False)
    print(f'{tag:6s} ckpt: d_model={c["config"]["d_model"]}, epoch={c["epoch"]}, '
          f'use_traj_branch={c["config"].get("use_traj_branch")}')
    assert c['config']['d_model'] == 256

cV2 = torch.load(V2_CKPT, map_location='cpu', weights_only=False)
print(f'V2     ckpt: d_model={cV2["config"]["d_model"]}, epoch={cV2["epoch"]}, '
      f'head_set={cV2["config"].get("head_set")}  -> warm-start backbone')
assert cV2['config']['d_model'] == 256
print('V2_CKPT:', V2_CKPT)

## 4. Train the critic / value decoder

`--head-set critic` selects `CrossEntityCriticModel` (V2 backbone + per-player `ValueDecoder`). `--init-from $V2_CKPT` warm-starts the backbone (`cross.*` / `consolidator.*`) from the V2 playerpair ckpt with `strict=False`; `--unfreeze l2 --backbone-lr-mult 0.1` trains `value_decoder.*` at full LR and `cross.*` at 0.1x LR while leaving `consolidator.*` frozen. `--aux-posterior` adds the future-T=10 teacher view (`t+45 ... t`) and consistency loss. `--cross-cache-path` points at the prebuilt cache; `train()` slices it into the 80/10/10 splits via the manifest. Loss = per-player win-rate BCE (ties excluded) + λ_rank·ListMLE on `final_rank` (λ_rank = 0.5), plus posterior/consistency aux terms when enabled.

In [ ]:
D_MODEL    = 256
WEIGHT_DECAY = 1e-4
SEED       = 1729
DEVICE     = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'cross_T10_critic_winrank_{DATASET}_d{D_MODEL}_b{BATCH_SIZE}_{EPOCHS}ep_lr{LR:g}_{TS}'
OUT_DIR = f'data/runs/cross_entity/{RUN_TAG}'
print('out dir:', OUT_DIR)

In [ ]:
!python -u -m agents.transformer_v2.pretrain.cross_entity \
  --train-mode frozen \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --planet-run-dir $PLANET_RUN_DIR \
  --entity-run-dir $ENTITY_RUN_DIR \
  --out-dir $OUT_DIR \
  --cross-cache-path $CACHE_PATH \
  --init-from $V2_CKPT \
  --unfreeze l2 \
  --backbone-lr-mult 0.1 \
  --aux-posterior \
  --lambda-cons 0.2 \
  --lambda-post 0.5 \
  --d-model $D_MODEL \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --head-set critic \
  --seed $SEED \
  --num-load-workers $NUM_LOAD_WORKERS \
  --num-workers $NUM_WORKERS \
  --device $DEVICE

## 5. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent], check=True)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{dst_parent}{src.name}/'], check=False)